# Part 6: Clustering

**Course:** 2026 KMITL Data Analytics

How can we discover groups when nobody has supplied the correct labels? This lesson builds clustering ideas from distances and a hand-worked example, then compares K-means, hierarchical clustering and DBSCAN. All datasets are small and created in memory; a CPU is enough.


## Learning objectives

By the end you can:
1. Distinguish clustering from classification and calculate two distances.
2. Assign points to centroids, update the centroids and explain inertia.
3. Use elbow and silhouette charts without treating them as proof of a true grouping.
4. Read a proximity matrix and dendrogram, and compare four linkage rules.
5. Explain DBSCAN core, border and noise points.
6. Choose between clustering methods based on shape, scale and noise.


## Required imports

NumPy handles small calculations, SciPy supplies distances and linkage, and scikit-learn supplies datasets and clustering. Matplotlib and Seaborn draw the results. Run all cells in order. No files or downloads are needed.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import cdist
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
print('Libraries loaded.')


## Dataset introduction

We use four small examples:

| Data | Size | Purpose |
|---|---|---|
| Hand coordinates | 6 points | Distances, assignments, centroids and linkage |
| Blobs | 240 points | Compact groups and choosing K |
| Density example | 6 points | Check core, border and noise exactly |
| Two moons with isolated points | 304 points | Curved groups and noise |

A **cluster** is a group of points that are similar under a chosen rule. **Classification** learns from known class labels; **clustering** looks for structure in unlabeled data. The dataset generators return labels, but we discard them: the algorithms must not use them. Cluster numbers are just names, not an order or a predicted class meaning. Different distance rules can produce different groups.


## 1. Similarity starts with distance

Take points A = (1, 1) and B = (4, 5). The horizontal difference is 3 and the vertical difference is 4. **Euclidean distance** is the straight-line length: square both differences, add them, then take the square root. Here that is the square root of 9 + 16, which is **5**. **Manhattan distance** adds absolute differences: 3 + 4 = **7**, like moving along streets in a grid. Smaller distance means more similar coordinates.

For points x and y with p features, j names a feature:

$$d_E(x,y)=\sqrt{\sum_{j=1}^{p}(x_j-y_j)^2},\qquad d_M(x,y)=\sum_{j=1}^{p}|x_j-y_j|.$$

A **proximity matrix** stores all pairwise distances. Its diagonal is zero, and these distance matrices are symmetric. A similarity matrix would instead put large values on similar pairs; always check which kind you have.


In [ ]:
point_a = np.array([1., 1.])
point_b = np.array([4., 5.])
euclidean = np.linalg.norm(point_a - point_b)
manhattan = np.abs(point_a - point_b).sum()
print('Euclidean:', euclidean, '| Manhattan:', manhattan)
assert np.isclose(euclidean, 5) and np.isclose(manhattan, 7)

small_points = np.array([[1., 1.], [1., 3.], [3., 2.], [7., 7.], [7., 9.], [9., 8.]])
point_names = list('ABCDEF')
distance_matrix = cdist(small_points, small_points, metric='euclidean')
print(pd.DataFrame(distance_matrix, index=point_names, columns=point_names).round(2))
assert np.allclose(distance_matrix, distance_matrix.T)
assert np.allclose(np.diag(distance_matrix), 0)


### Feature scale changes the question

A difference of 1 year and 1,000 baht is mostly a money difference under raw Euclidean distance. Standardization subtracts a feature's mean and divides by its standard deviation. If income is 30,000, its mean is 20,000 and its standard deviation is 5,000, the standardized value is (30,000 − 20,000) / 5,000 = **2**.

For a value x, feature mean m and feature standard deviation s, the standardized value is z:
$$z=(x-m)/s.$$

The blob example stretches its second coordinate by 100 to imitate different units. We then scale both features. Scaling is a modeling choice, not always an improvement: preserve meaningful physical distances when appropriate. We fit on all points for this descriptive analysis. For future-data evaluation, fit the scaler only on training data and reuse it.


In [ ]:
blob_points, _ = make_blobs(n_samples=240, centers=[[-4, -3], [0, 4], [4, -2]],
                            cluster_std=0.65, random_state=RANDOM_STATE)
raw_points = blob_points * np.array([1., 100.])
scaler = StandardScaler()
scaled_points = scaler.fit_transform(raw_points)
print(pd.DataFrame({'raw spread': raw_points.std(axis=0),
                    'scaled spread': scaled_points.std(axis=0)}, index=['feature 1', 'feature 2']))
assert np.allclose(scaled_points.mean(axis=0), 0, atol=1e-12)
assert np.allclose(scaled_points.std(axis=0), 1)


## 2. K-means: assign, then update

K is the number of clusters we ask for. A **centroid** is the coordinate-wise mean of the points assigned to one cluster. Start with K centers, assign each point to its nearest center, then move each center to its assigned mean. Repeat until assignments or centers stop changing. Standard K-means uses squared Euclidean distance, not Manhattan distance.

Our six points start with centers (1, 1) and (7, 7). The first three join center 1; the last three join center 2. The new first center is ((1 + 1 + 3)/3, (1 + 3 + 2)/3) = **(1.67, 2)**. The second becomes **(7.67, 8)**.

Let C_k be the set of points assigned to cluster k and |C_k| its size. The updated center is mu_k:
$$\mu_k=\frac{1}{|C_k|}\sum_{x_i\in C_k}x_i.$$

**Inertia** adds the squared distance from every point to its assigned center. Distances 1, 2 and 3 contribute 1 + 4 + 9 = **14**. With n points, point x_i and its assigned cluster c_i, J is inertia and double bars mean Euclidean length:
$$J=\sum_{i=1}^{n}\|x_i-\mu_{c_i}\|^2.$$

Lower inertia is better for the same data and K, but simply increasing K usually lowers it too. Initial centers matter: K-means can settle at a local solution. The library uses K-means++ initialization and we request 10 starts. An empty cluster has no mean; our tiny demonstration checks that none is empty, while the library handles this during real fitting.


In [ ]:
start_centers = np.array([[1., 1.], [7., 7.]])
small_labels = cdist(small_points, start_centers).argmin(axis=1)
assert set(small_labels) == {0, 1}
new_centers = np.array([small_points[small_labels == group].mean(axis=0) for group in range(2)])
old_inertia = ((small_points - start_centers[small_labels]) ** 2).sum()
new_inertia = ((small_points - new_centers[small_labels]) ** 2).sum()
assert np.allclose(new_centers, [[5 / 3, 2], [23 / 3, 8]])
assert new_inertia <= old_inertia
print('Updated centers:', new_centers.round(2))
print(f'Inertia: {old_inertia:.2f} -> {new_inertia:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, centers, title in zip(axes, [start_centers, new_centers], ['1. Assign to starting centers', '2. Update centers to means']):
    for group in range(2):
        members = small_points[small_labels == group]
        ax.scatter(members[:, 0], members[:, 1], label=f'Cluster {group}', s=70)
    ax.scatter(centers[:, 0], centers[:, 1], marker='X', c='black', s=180, label='Centroids')
    for point, group, name in zip(small_points, small_labels, point_names):
        ax.plot([point[0], centers[group, 0]], [point[1], centers[group, 1]], 'k:', alpha=0.4)
        ax.annotate(name, point + 0.12)
    ax.set(title=title, xlabel='Coordinate 1', ylabel='Coordinate 2', xlim=(0, 10), ylim=(0, 10))
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


**Read the two panels:** dotted lines show assignments. The black crosses move into the middle of their groups, reducing inertia from 18 to about 9.33. Assigning again leaves these groups unchanged. Real data can require many updates, and one update does not generally finish the algorithm.


In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE)
blob_labels = kmeans.fit_predict(scaled_points)
fig, ax = plt.subplots(figsize=(6, 5))
for group in range(3):
    members = scaled_points[blob_labels == group]
    ax.scatter(members[:, 0], members[:, 1], s=25, label=f'Cluster {group}')
ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           marker='X', c='black', s=180, label='Centroids')
ax.set(title='K-means on standardized blobs', xlabel='Scaled feature 1', ylabel='Scaled feature 2')
ax.legend()
plt.tight_layout()
plt.show()
print('Inertia:', round(kmeans.inertia_, 2))


**Read the scatter plot:** three compact clouds surround their centers. This is a favorable shape for K-means. The axes are standard deviations, not the original units. The colors show discovered groups; they do not prove that three meaningful classes exist in an application.


## 3. Choosing K: elbow and silhouette

The **elbow method** plots inertia against K. For example, inertias 100, 40, 15, 13 show a large improvement up to K = 3, then a small gain. That bend suggests trying three groups; some datasets have no clear elbow.

The **silhouette score** compares cohesion (closeness within a cluster) with separation from the nearest other cluster. For one point, let a be its mean distance to other points in its own cluster. Let b be the smallest mean distance to the points in any other cluster. If a = 2 and b = 6, its score is (6 − 2) / 6 = **0.67**. In general the score s is:
$$s=(b-a)/\max(a,b).$$

Scores lie between −1 and 1: near 1 means well separated, near 0 means near a boundary, and negative means another cluster may fit better. The overall score averages the point scores. It requires at least two clusters and fewer clusters than points; a singleton's score is defined as zero by scikit-learn. Silhouette favors some compact shapes and is not universal evidence of usefulness.


In [ ]:
rows = []
for k in range(1, 8):
    model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(scaled_points)
    score = silhouette_score(scaled_points, model.labels_) if k > 1 else np.nan
    rows.append({'K': k, 'inertia': model.inertia_, 'silhouette': score})
k_results = pd.DataFrame(rows)
print(k_results.round(3).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(k_results['K'], k_results['inertia'], 'o-', label='Inertia')
axes[0].set(title='Elbow method', xlabel='Number of clusters K', ylabel='Sum of squared distances')
axes[1].plot(k_results['K'], k_results['silhouette'], 'o-', label='Silhouette')
axes[1].set(title='Separation and cohesion', xlabel='Number of clusters K', ylabel='Mean silhouette')
for ax in axes:
    ax.set_xticks(range(1, 8))
    ax.legend()
plt.tight_layout()
plt.show()


**Read the curves:** the inertia curve bends around three groups and the silhouette chart supports this choice on these generated blobs. K = 1 has no silhouette value, not a score of zero. A larger K can reduce inertia while breaking useful groups apart. Combine these charts with domain knowledge, stability across seeds and the purpose of the analysis.


## 4. Hierarchical clustering and linkage

A hierarchy represents groups inside larger groups. **Agglomerative clustering** starts with one point per group and repeatedly merges the closest pair of groups. **Divisive clustering** starts with all points together and repeatedly splits groups; it is a different, top-down process, not simply a reversed list of merges. We implement the common bottom-up method here.

A linkage rule defines distance between groups. Consider one-dimensional groups A = {0, 2} and B = {5, 9}. Their cross-group distances are 5, 9, 3 and 7:

| Rule | Meaning | Numerical example |
|---|---|---|
| Single | Closest pair | minimum = 3 |
| Complete | Farthest pair | maximum = 9 |
| Average | Mean of all cross-group distances | (5 + 9 + 3 + 7)/4 = 6 |
| Centroid | Distance between group means | means 1 and 7; distance = 6 |

Let d(x,y) be point distance, A and B be groups, and mu_A and mu_B be their means. |A| and |B| count group members. The four distances are:
$$d_{single}=\min_{x\in A,y\in B}d(x,y),\quad d_{complete}=\max_{x\in A,y\in B}d(x,y),$$
$$d_{average}=\frac{\sum_{x\in A}\sum_{y\in B}d(x,y)}{|A||B|},\quad d_{centroid}=\|\mu_A-\mu_B\|.$$

Single linkage can join long chains; complete linkage favors compact groups; average uses all pair distances. Centroid linkage uses Euclidean geometry and can have **inversions**, where a later merge is drawn below an earlier one. A **dendrogram** draws the merge history: leaves are observations and merge height is the linkage distance. For monotonic linkage methods, cutting horizontally gives clusters below that height. We pass coordinates to SciPy, not a square distance matrix that could be misread as features.


In [ ]:
group_a = np.array([[0.], [2.]])
group_b = np.array([[5.], [9.]])
cross_distances = cdist(group_a, group_b)
linkage_values = [cross_distances.min(), cross_distances.max(), cross_distances.mean(),
                  np.linalg.norm(group_a.mean(axis=0) - group_b.mean(axis=0))]
assert np.allclose(linkage_values, [3, 9, 6, 6])
print(dict(zip(['single', 'complete', 'average', 'centroid'], linkage_values)))
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, method in zip(axes.flat, ['single', 'complete', 'average', 'centroid']):
    tree = linkage(small_points, method=method, metric='euclidean')
    dendrogram(tree, labels=point_names, color_threshold=0, above_threshold_color='steelblue', ax=ax)
    ax.set(title=f'{method.title()} linkage', xlabel='Point', ylabel='Merge distance')
plt.tight_layout()
plt.show()


**Read the dendrograms:** A–C merge into one small group and D–F into another before the final joining step. The final merge heights differ because the linkage rules ask different questions. Leaf order is for drawing, not a numeric ranking, and horizontal spacing does not measure distance. This small example has clear groups; real hierarchies can be less clear. Pairwise distance calculations can also become expensive for large datasets.


## 5. DBSCAN: density, borders and noise

DBSCAN grows clusters through dense neighborhoods rather than choosing K. **eps** is a distance radius and **min_samples** is the minimum number of points in that radius, including the point itself in scikit-learn. A **core point** has enough neighbors. A **border point** is not core but lies within eps of a core point. A **noise point** is neither; scikit-learn gives it label −1. Clusters expand through core points; a border point does not continue expansion by itself.

On a line, use points 0, 0.05, 0.10, 0.15, 0.34 and 1.0. With eps = 0.20 and min_samples = 4, the first four are core. The point 0.34 reaches the core point 0.15 but has too few neighbors itself, so it is border. The point 1.0 is isolated noise.

Too small an eps can mark nearly everything as noise; too large can merge different groups. Scaling and varying density matter. A noise label means unusual under these settings, not necessarily an error or fraud. DBSCAN does not provide a standard `predict` method for future points.


In [ ]:
line_points = np.array([[0.], [0.05], [0.10], [0.15], [0.34], [1.]])
line_model = DBSCAN(eps=0.20, min_samples=4).fit(line_points)
line_core = np.zeros(len(line_points), dtype=bool)
line_core[line_model.core_sample_indices_] = True
line_type = np.where(line_core, 'core', np.where(line_model.labels_ == -1, 'noise', 'border'))
print(pd.DataFrame({'position': line_points[:, 0], 'label': line_model.labels_, 'type': line_type}))
assert line_type.tolist() == ['core', 'core', 'core', 'core', 'border', 'noise']


## 6. Compare the algorithms on curved data

The two-moons data has curved groups. We add four distant points deliberately, keep both coordinates in their original comparable units, and compare K-means, single-linkage agglomerative clustering and DBSCAN on exactly the same input. Choosing two clusters for the first two methods is an illustration, not knowledge they discover. DBSCAN uses eps = 0.20 and min_samples = 5; these are teaching settings, not universal defaults.


In [ ]:
moon_points, _ = make_moons(n_samples=300, noise=0.05, random_state=RANDOM_STATE)
moon_points = np.vstack([moon_points, [[-1.8, 1.8], [2.8, 1.8], [-1.8, -1.3], [2.8, -1.3]]])
moon_kmeans = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE).fit_predict(moon_points)
moon_hierarchy = AgglomerativeClustering(n_clusters=2, linkage='single').fit_predict(moon_points)
density_model = DBSCAN(eps=0.20, min_samples=5).fit(moon_points)
moon_density = density_model.labels_
core_mask = np.zeros(len(moon_points), dtype=bool)
core_mask[density_model.core_sample_indices_] = True

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
comparison = [('K-means', moon_kmeans), ('Single-linkage hierarchy', moon_hierarchy), ('DBSCAN', moon_density)]
comparison_rows = []
for ax, (name, labels) in zip(axes, comparison):
    for group in sorted(set(labels) - {-1}):
        members = labels == group
        color = plt.get_cmap('tab10')(int(group) % 10)
        if name == 'DBSCAN':
            core = members & core_mask
            border = members & ~core_mask
            ax.scatter(*moon_points[core].T, s=20, color=color, label=f'Cluster {group}: core')
            ax.scatter(*moon_points[border].T, s=55, facecolors='none', edgecolors=[color], label=f'Cluster {group}: border')
        else:
            ax.scatter(*moon_points[members].T, s=20, color=color, label=f'Cluster {group}')
    noise = labels == -1
    if noise.any():
        ax.scatter(*moon_points[noise].T, c='black', marker='x', s=60, label='Noise')
    ax.set(title=name, xlabel='Coordinate 1', ylabel='Coordinate 2')
    ax.legend(fontsize=7)
    ax.set_aspect('equal')
    retained = ~noise
    groups = len(set(labels[retained]))
    score = silhouette_score(moon_points[retained], labels[retained]) if 1 < groups < retained.sum() else np.nan
    comparison_rows.append({'method': name, 'clusters': groups, 'noise': int(noise.sum()),
                            'retained fraction': retained.mean(), 'silhouette (non-noise)': score})
plt.tight_layout()
plt.show()
print(pd.DataFrame(comparison_rows).round(3).to_string(index=False))
assert np.all(moon_density[-4:] == -1)


**Read the comparison:** K-means divides space into compact regions and can cut across a moon. Single linkage can follow a curved chain, but isolated points can consume clusters when we force a cut at K = 2. DBSCAN can trace the two dense curves and leave the distant crosses as noise. Filled dots are core points; hollow circles are border points if any are present.

**Read the table carefully:** DBSCAN's silhouette excludes noise, while the other methods assign every point. These scores therefore do not evaluate identical subsets. Report the retained fraction alongside the score; a method should not win merely by discarding hard points. Even on the same subset, Euclidean silhouette may prefer compact groups over meaningful curves. Noise removal and algorithm choice should reflect the goal, not just the largest score.

| Method | Useful when | Important limit |
|---|---|---|
| K-means | Compact, similarly sized groups; a fast baseline | Must choose K; sensitive to outliers and scale |
| Hierarchical | A nested view and merge history are useful | Linkage matters; pairwise work can be costly |
| DBSCAN | Dense irregular shapes and noise are expected | eps is scale-sensitive; varying densities are difficult |


## Common mistakes

- Calling cluster numbers known class labels, or expecting the same numbers after every run.
- Letting a large-unit feature dominate distance without considering scaling.
- Choosing the largest K because inertia is smallest.
- Using one silhouette value as proof of a useful business grouping.
- Passing a square proximity matrix to `linkage` as though it were coordinates.
- Confusing DBSCAN border points with noise, or forgetting the point itself counts toward min_samples.
- Comparing scores after dropping different sets of points without reporting coverage.
- Assuming every method has a prediction rule for new observations. K-means does; these hierarchical and DBSCAN estimators do not provide one by default.


In [ ]:
assert scaled_points.shape == (240, 2)
assert len(set(blob_labels)) == 3
assert np.isclose(kmeans.inertia_, ((scaled_points - kmeans.cluster_centers_[blob_labels]) ** 2).sum())
assert k_results.loc[k_results['K'] > 1, 'silhouette'].between(-1, 1).all()
assert len(moon_density) == len(moon_points)
print('Clustering lesson checks passed.')


## Summary

Clustering finds structure without training labels. Distance and scaling define what similar means. K-means alternates assignments and mean updates; elbow and silhouette charts help assess K but do not decide usefulness. Hierarchical methods build a nested merge history using a chosen linkage rule. DBSCAN uses density to distinguish core, border and noise points. Compare the shapes, retained data and practical meaning of the groups, not just a single metric.
